In [13]:
# %% Setup: Ne2ra el clean event log w n3mel mapping l el activities
import csv
from collections import defaultdict
from itertools import combinations

# el mapping elly fel README b el tartib el abgady
act_map = {
    'Assign Advisor': 'a',
    'Check Eligibility': 'b',
    'Confirm Enrollment': 'c',
    'Issue Student Card': 'd',
    'Pay Tuition': 'e',
    'Select Courses': 'f',
    'Submit Application': 'g',
    'Verify Documents': 'h'
}

file_path = '/content/Q2_clean_event_log.csv'

with open(file_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = list(reader)

# bngm3 kol case b el activities bta3etha b tartibaha (traces)
cases = defaultdict(list)
for r in rows:
    cid = r[0].strip()
    act = r[1].strip()
    cases[cid].append(act_map[act])

traces = [tuple(acts) for acts in cases.values()]

In [14]:
# @title Q2.1
# 3adad el distinct traces (variants)
distinct_traces = set(traces)
q2_1 = len(distinct_traces)
print(f"Q2.1: {q2_1}")

Q2.1: 8


In [15]:
# @title Q2.2
# 3adad el pairs (x, y) elly feha x directly follows y (>L)
directly_follows = set()
for t in traces:
    for i in range(len(t) - 1):
        directly_follows.add((t[i], t[i+1]))

q2_2 = len(directly_follows)
print(f"Q2.2: {q2_2}")

Q2.2: 12


In [16]:
# @title Q2.3, Q2.4, Q2.5
# 3adad el unordered pairs (28 pairs) fe causality, parallel, w hash
all_acts = sorted(list(act_map.values()))
pairs_28 = list(combinations(all_acts, 2))

causality_pairs = []
parallel_pairs = []
hash_pairs = []
footprint = {}

for x, y in pairs_28:
    x_follows_y = (x, y) in directly_follows
    y_follows_x = (y, x) in directly_follows
    if x_follows_y and not y_follows_x:
        footprint[(x, y)] = '→'
        causality_pairs.append((x, y))
    elif y_follows_x and not x_follows_y:
        footprint[(x, y)] = '←'
        causality_pairs.append((x, y))
    elif x_follows_y and y_follows_x:
        footprint[(x, y)] = '||'
        parallel_pairs.append((x, y))
    else:
        footprint[(x, y)] = '#'
        hash_pairs.append((x, y))

# Q2.3: causality pairs (x -> y aw y -> x)
q2_3 = len(causality_pairs)
print(f"Q2.3: {q2_3}")

# Q2.4: parallel pairs (x || y)
q2_4 = len(parallel_pairs)
print(f"Q2.4: {q2_4}")

# Q2.5: hash pairs (x # y)
q2_5 = len(hash_pairs)
print(f"Q2.5: {q2_5}")

Q2.3: 10
Q2.4: 1
Q2.5: 17


In [17]:
# @title Q2.6 & Q2.7
# start activities (TI) w 3adadhom |TI|
TI = sorted(list(set(t[0] for t in traces)))
q2_6 = len(TI)
q2_7 = ','.join(TI)
print(f"Q2.6: {q2_6}")
print(f"Q2.7: {q2_7}")

Q2.6: 1
Q2.7: g


In [18]:
# @title Q2.8 & Q2.9
# end activities (TO) w 3adadhom |TO|
TO = sorted(list(set(t[-1] for t in traces)))
q2_8 = len(TO)
q2_9 = ','.join(TO)
print(f"Q2.8: {q2_8}")
print(f"Q2.9: {q2_9}")

Q2.8: 2
Q2.9: c,d


In [19]:
# @title Q2.10
# 3adad el pairs (A, B) fe step 4 (|XL|)
def is_hash(x, y):
    return (x, y) not in directly_follows and (y, x) not in directly_follows

def is_causal(x, y):
    return (x, y) in directly_follows and (y, x) not in directly_follows

# kol el non-empty subsets elly el 3anasr bta3etha pairwise #
subsets = []
for r in range(1, len(all_acts) + 1):
    for s in combinations(all_acts, r):
        if all(is_hash(x1, x2) for x1, x2 in combinations(s, 2)):
            subsets.append(set(s))

XL = []
for A in subsets:
    for B in subsets:
        if all(is_causal(a, b) for a in A for b in B):
            XL.append((frozenset(A), frozenset(B)))

q2_10 = len(XL)
print(f"Q2.10: {q2_10}")

Q2.10: 14


In [20]:
# @title Q2.11
# 3adad el maximal pairs fe step 5 (|YL|)
YL = []
for A, B in XL:
    is_maximal = True
    for A_p, B_p in XL:
        if (A, B) != (A_p, B_p) and A.issubset(A_p) and B.issubset(B_p):
            is_maximal = False
            break
    if is_maximal:
        YL.append((A, B))

q2_11 = len(YL)
print(f"Q2.11: {q2_11}")

Q2.11: 6


In [21]:
# @title Q2.12
# 3adad el places fe alpha(L): |YL| + source (iL) + sink (oL)
q2_12 = len(YL) + 2
print(f"Q2.12: {q2_12}")

Q2.12: 8


In [22]:
# @title Q2.13
# 3adad el arcs fe FL
# arcs mn A l el place + mn el place l B + source l TI + TO l sink
arcs = sum(len(A) + len(B) for A, B in YL) + len(TI) + len(TO)
q2_13 = arcs
print(f"Q2.13: {q2_13}")

Q2.13: 19


In [23]:
# @title Q2.14
# Footprint matrix (el 28 pairs fo2 el diagonal)
print("Q2.14 Footprint Matrix (28 pairs):")
for x, y in pairs_28:
    print(f"{x} vs {y}: {footprint[(x, y)]}")

Q2.14 Footprint Matrix (28 pairs):
a vs b: ←
a vs c: #
a vs d: #
a vs e: →
a vs f: ||
a vs g: #
a vs h: ←
b vs c: #
b vs d: #
b vs e: #
b vs f: →
b vs g: ←
b vs h: #
c vs d: #
c vs e: ←
c vs f: #
c vs g: #
c vs h: #
d vs e: ←
d vs f: #
d vs g: #
d vs h: #
e vs f: ←
e vs g: #
e vs h: #
f vs g: #
f vs h: ←
g vs h: →


In [24]:
# %% Q2.14
# footprint matrix fe shakl 2D grid/table
def get_rel(x, y):
  if x == y:
    return '#'
  x_fol = (x, y) in directly_follows
  y_fol = (y, x) in directly_follows
  if x_fol and not y_fol:
    return '→'
  elif y_fol and not x_fol:
    return '←'
  elif x_fol and y_fol:
    return '||'
  else:
    return '#'


# netba3 el matrix b header w rows
print('      ' + '    '.join(all_acts))
print('   ' + '-' * 38)
for x in all_acts:
  row = [get_rel(x, y).center(3) for y in all_acts]
  print(f' {x} | ' + '  '.join(row))

      a    b    c    d    e    f    g    h
   --------------------------------------
 a |  #    ←    #    #    →    ||   #    ← 
 b |  →    #    #    #    #    →    ←    # 
 c |  #    #    #    #    ←    #    #    # 
 d |  #    #    #    #    ←    #    #    # 
 e |  ←    #    →    →    #    ←    #    # 
 f |  ||   ←    #    #    →    #    #    ← 
 g |  #    →    #    #    #    #    #    → 
 h |  →    #    #    #    #    →    ←    # 
